# Cumberland CVT Diagnostics Workflow

This notebook is focused specifically on the new constrained full-seed CVT/Lloyd implementation.

It compares four grid states against the same Cumberland-style source inputs:

- `baseline`
- `cleanup_only`
- `experimental_cvt`
- optional imported `AlgoMesh DISU`

Use this notebook to inspect whether the new CVT implementation is actually improving the mesh, or safely rejecting bad iterations and falling back.


## Important Notes

- `slopes.gpkg` is still off by default because it is the Cumberland input most likely to crash `triangle.exe`.
- the old legacy point refinement is also off by default because it makes the whole domain much denser.
- `experimental_cvt` is still experimental. The point of this notebook is to inspect it carefully, not to assume it is better.


In [ ]:
from pathlib import Path
import sys
import time
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "myflopy" / "__init__.py").exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError("Could not locate the myflopy repo root from the current notebook working directory.")
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
for module_name in list(sys.modules):
    if module_name == "myflopy" or module_name.startswith("myflopy."):
        del sys.modules[module_name]

import myflopy as mf
mf = importlib.reload(mf)
print(f"Using myflopy from: {mf.__file__}")

cumberland_root = Path.home() / "mf6/Cumberland general"
boundary_root = cumberland_root / "Boundaries"
herrera_root = cumberland_root / "Herrera"
algomesh_root = cumberland_root / "algomesh"
surface_root = cumberland_root / "Surfaces"

domain_path = cumberland_root / "domain_v3.gpkg"
slopes_path = boundary_root / "slopes.gpkg"
deep_lake_path = boundary_root / "deep lake.gpkg"
hyde_lake_path = boundary_root / "hyde lake.gpkg"
deep_creek_path = boundary_root / "deep creek.gpkg"
hyde_creek_path = boundary_root / "hyde creek.gpkg"
sw_facilities_path = herrera_root / "Infiltration_Facilities.gpkg"
disu_path = algomesh_root / "cumb_vor_algomesh_facilities_rev2.1.disu"
top_raster = surface_root / "top_of_model_with_pits_no_overlap_with_botm_v5.tif"
botm_raster = surface_root / "cumb_aq_btm_v14c.tif"

required_paths = [
    domain_path,
    deep_lake_path,
    hyde_lake_path,
    deep_creek_path,
    hyde_creek_path,
    sw_facilities_path,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing Cumberland inputs:\n" + "\n".join(str(path) for path in missing))

workspace = Path.cwd().resolve().parents[1] / "artifacts" / "cumberland_cvt_diagnostics"
workspace.mkdir(parents=True, exist_ok=True)

background_max_area = 40000
deep_creek_max_area = 200
hyde_creek_max_area = 200
lake_max_area = 2000
facility_max_area = 500

include_slopes_refinement = False
include_legacy_point_region = False
legacy_point_region_max_area = 2000
include_algomesh_comparison = disu_path.exists()
run_experimental_cvt = True

workspace


In [ ]:
def build_cumberland_triangle(mesh_name: str, *, mode: str = "baseline", include_slopes: bool = False):
    tri = mf.TriangleGrid(model_ws=str(workspace / mesh_name), angle=30)

    tri.set_domain_file(
        domain_path,
        buffer=-1,
        simplify_tolerance=10,
        densify_dist=25,
        max_area=background_max_area,
        label="domain",
    )
    tri.add_line_feature(
        deep_creek_path,
        buffer=20,
        simplify_tolerance=10,
        densify_dist=100,
        max_area=deep_creek_max_area,
        label="deep_creek",
        priority=4,
    )
    tri.add_line_feature(
        hyde_creek_path,
        buffer=20,
        simplify_tolerance=10,
        densify_dist=100,
        max_area=hyde_creek_max_area,
        label="hyde_creek",
        priority=4,
    )
    tri.add_region_file(
        deep_lake_path,
        simplify_tolerance=20,
        max_area=lake_max_area,
        label="deep_lake",
        priority=5,
    )
    tri.add_region_file(
        hyde_lake_path,
        simplify_tolerance=20,
        max_area=lake_max_area,
        label="hyde_lake",
        priority=5,
    )

    sw_facilities = mf.read_shp_gpkg(sw_facilities_path)
    for idx, poly in enumerate(sw_facilities.geometry):
        tri.add_region_polygon(
            poly,
            densify_dist=50,
            max_area=facility_max_area,
            buffer=-5,
            label=f"facility_{idx}",
            priority=6,
            source="facility",
        )

    if include_slopes:
        tri.add_region_file(
            slopes_path,
            max_area=4000,
            densify_dist=200,
            buffer=100,
            label="slopes",
            priority=2,
        )

    if include_legacy_point_region:
        tri.add_region(point=(1363943, 112693), maximum_area=legacy_point_region_max_area)

    start = time.perf_counter()
    if mode == "cleanup_only":
        tri.clean_geometry(
            simplify_tolerance=2,
            target_segment_length=150,
            resample_region_sources=("line",),
        )
        tri.build(verbose=False)
        quality = tri.quality_report(include_voronoi=True)
        report = {"cleanup": tri._last_cleanup_report, "optimization": None, "quality": quality}
    elif mode == "experimental_cvt":
        report = tri.build_mesh(
            profile="balanced",
            protect_sources=("line",),
            protected_labels=["deep_creek", "hyde_creek", "deep_lake", "hyde_lake"],
            simplify_tolerance=2,
            target_segment_length=150,
            optimization_iterations=3,
            verbose=True,
        )
        quality = report["quality"]
    else:
        tri.build(verbose=False)
        quality = tri.quality_report(include_voronoi=True)
        report = {"cleanup": None, "optimization": None, "quality": quality}

    report["elapsed_seconds"] = time.perf_counter() - start
    return tri, report


In [ ]:
tri_base, base_report = build_cumberland_triangle("triangle_baseline", mode="baseline", include_slopes=include_slopes_refinement)
tri_clean, clean_report = build_cumberland_triangle("triangle_cleanup_only", mode="cleanup_only", include_slopes=include_slopes_refinement)

experimental_result = None
if run_experimental_cvt:
    tri_cvt, cvt_report = build_cumberland_triangle("triangle_experimental_cvt", mode="experimental_cvt", include_slopes=include_slopes_refinement)
    experimental_result = (tri_cvt, cvt_report)
    if cvt_report["optimization"]["status"] == "fallback_original_mesh":
        print("Experimental CVT candidate was rejected by the quality gates, so the final mesh was rolled back to the cleanup-only state.")
else:
    cvt_report = None

comparison = {
    "baseline": pd.Series(base_report["quality"]),
    "cleanup_only": pd.Series(clean_report["quality"]),
}
if cvt_report is not None:
    comparison["experimental_cvt"] = pd.Series(cvt_report["quality"])

comparison_df = pd.DataFrame(comparison)
comparison_df.loc[[
    "num_vertices",
    "num_triangles",
    "duplicate_vertex_count",
    "zero_area_triangle_count",
    "tiny_triangle_count",
    "sliver_triangle_count",
    "triangle_angle_min_overall",
    "triangle_edge_ratio_mean",
    "neighbor_area_ratio_mean",
    "voronoi_status",
    "voronoi_cell_count",
]]


In [ ]:
timing = {
    "baseline": base_report["elapsed_seconds"],
    "cleanup_only": clean_report["elapsed_seconds"],
}
if cvt_report is not None:
    timing["experimental_cvt"] = cvt_report["elapsed_seconds"]
pd.Series(timing)


In [ ]:
vor_base = mf.VoronoiGridPlus(tri_base, rasters=[top_raster, botm_raster], crs="EPSG:2926", name="cumberland_base")
vor_clean = mf.VoronoiGridPlus(tri_clean, rasters=[top_raster, botm_raster], crs="EPSG:2926", name="cumberland_cleanup_only")
vor_cvt = None if experimental_result is None else mf.VoronoiGridPlus(experimental_result[0], rasters=[top_raster, botm_raster], crs="EPSG:2926", name="cumberland_experimental_cvt")
cvt_status = None if cvt_report is None else cvt_report["optimization"]["status"]

fig, axes = plt.subplots(1, 3 if vor_cvt is not None else 2, figsize=(22 if vor_cvt is not None else 16, 8), constrained_layout=True)
axes = np.atleast_1d(axes)

vor_base.gdf_vorPolys.boundary.plot(ax=axes[0], linewidth=0.15, color="black")
axes[0].set_title(f"Baseline\nCells: {vor_base.ncpl:,}")
axes[0].set_aspect("equal")

vor_clean.gdf_vorPolys.boundary.plot(ax=axes[1], linewidth=0.15, color="black")
axes[1].set_title(f"Cleanup-only\nCells: {vor_clean.ncpl:,}")
axes[1].set_aspect("equal")

if vor_cvt is not None:
    vor_cvt.gdf_vorPolys.boundary.plot(ax=axes[2], linewidth=0.15, color="black")
    if cvt_status == "fallback_original_mesh":
        axes[2].set_title(f"Experimental CVT\nRejected -> fallback to cleanup-only\nCells: {vor_cvt.ncpl:,}")
    else:
        axes[2].set_title(f"Experimental CVT\nStatus: {cvt_status}\nCells: {vor_cvt.ncpl:,}")
    axes[2].set_aspect("equal")

plt.show()


In [ ]:
algomesh_summary = None
if include_algomesh_comparison:
    vor_algomesh = mf.VoronoiGridPlus.vor_from_disu(
        disu_path=disu_path,
        rasters=[top_raster, botm_raster],
        crs="EPSG:2926",
        name="cumberland_algomesh",
    )
    rows = [
        {
            "grid": "baseline",
            "cell_count": vor_base.ncpl,
            "mean_cell_area": float(np.mean(vor_base.get_cell_areas())),
        },
        {
            "grid": "cleanup_only",
            "cell_count": vor_clean.ncpl,
            "mean_cell_area": float(np.mean(vor_clean.get_cell_areas())),
        },
    ]
    if vor_cvt is not None:
        rows.append(
            {
                "grid": "experimental_cvt",
                "cell_count": vor_cvt.ncpl,
                "mean_cell_area": float(np.mean(vor_cvt.get_cell_areas())),
            }
        )
    rows.append(
        {
            "grid": "algomesh_disu",
            "cell_count": vor_algomesh.ncpl,
            "mean_cell_area": float(np.mean(vor_algomesh.get_cell_areas())),
        }
    )
    algomesh_summary = pd.DataFrame(rows)

algomesh_summary


In [ ]:
if cvt_report is not None:
    cvt_summary = pd.Series(cvt_report["optimization"])
    if cvt_report["optimization"]["status"] == "fallback_original_mesh":
        print("Because the candidate CVT mesh was rejected, the final plotted CVT mesh should match the cleanup-only mesh.")
    cvt_summary
else:
    print("Experimental CVT run was skipped.")
